# Discrete-time multinomial competing-risk model

This notebook estimates a stage-specific discrete-time multinomial model with three mutually exclusive project-year outcomes:

- **No transition**
- **Progress**
- **Failure**

`No transition` is the reference outcome. The multinomial specification guarantees that the three annual outcome probabilities are non-negative and sum to one. This makes the model directly suitable for cumulative-incidence calculations in which progression and failure are competing events.

The same stage-specific risk sets, duration variable, grouped end-use controls, macro controls and year fixed effects used in the primary progression/failure analysis are retained. A second specification replaces the grouped end-use indicators with `any_enduse`.

Inference uses a project-clustered sandwich covariance matrix constructed from project-level sums of the multinomial score contributions. The fitted models are saved once and consumed by Supplementary Figures S3–S5.

Run after Data Preparation. The country/macro specification uses the governance control selected in the master pipeline: `Governance_Score`, `Ease_of_doing_business`, or no additional governance control.


In [1]:
# Use the same project/data directory in standalone and master-run execution.
# With H2_PROJECT_DIR unset, run from the folder containing Data/ and the RDS files.
project_dir <- Sys.getenv("H2_PROJECT_DIR", unset = getwd())
project_dir <- normalizePath(project_dir, winslash = "/", mustWork = TRUE)
setwd(project_dir)

# The master pipeline passes this setting to each model notebook.
# Standalone execution defaults to Governance_Score.
governance_control <- Sys.getenv(
  "H2_GOVERNANCE_CONTROL",
  unset = "Governance_Score"
)

valid_governance_controls <- c(
  "Governance_Score",
  "Ease_of_doing_business",
  "none"
)

if (!governance_control %in% valid_governance_controls) {
  stop(
    paste0(
      "governance_control must be one of: ",
      paste(valid_governance_controls, collapse = ", ")
    )
  )
}


In [2]:
# ==============================================================================
# DISCRETE-TIME MULTINOMIAL COMPETING-RISK MODEL
# ==============================================================================

required_packages <- c(
  "dplyr",
  "tidyr",
  "purrr",
  "stringr",
  "tibble",
  "MASS",
  "nnet"
)

missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_packages) > 0) {
  stop(
    paste0(
      "Missing packages: ",
      paste(missing_packages, collapse = ", "),
      ". Install them before running this notebook."
    )
  )
}

invisible(lapply(required_packages, library, character.only = TRUE))
set.seed(123)

model_data_file <- "cloglog_data_processed.rds"

if (!file.exists(model_data_file)) {
  stop(
    paste0(
      "The model-ready data file '",
      model_data_file,
      "' was not found. Run Data Preparation before this notebook."
    )
  )
}

cloglog_data <- readRDS(model_data_file)

# ------------------------------------------------------------------------------
# Model specifications
# ------------------------------------------------------------------------------

enduse_controls_grouped <- c(
  "enduse_industrial",
  "enduse_transport",
  "enduse_chemicals",
  "enduse_power",
  "enduse_other"
)

enduse_controls_grouped <- enduse_controls_grouped[
  enduse_controls_grouped %in% names(cloglog_data)
]

if (length(enduse_controls_grouped) == 0) {
  stop("No grouped end-use variables were found in the model data.")
}

if (!"any_enduse" %in% names(cloglog_data)) {
  stop("The any_enduse variable is missing. Run Data Preparation first.")
}

core_controls <- c(
  "cap.mwel_log",
  "prev_time_in_status",
  "technology",
  "electricity_source"
)

governance_controls <- if (identical(governance_control, "none")) {
  character(0)
} else {
  governance_control
}

if (
  length(governance_controls) > 0L &&
  !governance_control %in% names(cloglog_data)
) {
  stop(
    paste0(
      "The selected governance control '",
      governance_control,
      "' is not available in cloglog_data_processed.rds."
    )
  )
}

macro_controls <- c(
  "Country_Risk_Spread_log",
  "RISE",
  "region",
  governance_controls
)

core_controls <- core_controls[core_controls %in% names(cloglog_data)]
macro_controls <- macro_controls[macro_controls %in% names(cloglog_data)]

grouped_model_name <- "Core + macro controls + year FE"
any_enduse_model_name <- "Core + macro controls + year FE + any enduse"

competing_control_steps <- list(
  "Core + macro controls + year FE" = c(
    core_controls,
    enduse_controls_grouped,
    macro_controls,
    "factor(year)"
  ),
  "Core + macro controls + year FE + any enduse" = c(
    core_controls,
    macro_controls,
    "factor(year)",
    "any_enduse"
  )
)

# Both specifications within a stage use the same complete-case sample. This
# keeps differences between the grouped and aggregate end-use specifications
# from being driven by changing observations.
common_controls <- unique(unlist(competing_control_steps, use.names = FALSE))

stage_specs <- list(
  "Concept -> Progress" = 1L,
  "Feasibility -> Progress" = 2L,
  "FID -> Progress" = 3L
)



Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union





Attaching package: ‘MASS’




The following object is masked from ‘package:dplyr’:

    select




## Multinomial estimation and project-clustered inference

For each origin stage, every project-year contributes exactly one categorical outcome. The multinomial logit model estimates the annual probabilities of remaining in the stage, progressing, or failing in one coherent probability system.

Project-clustered uncertainty is calculated from cluster sums of the observation-level multinomial score contributions. The resulting covariance matrix is used for the coefficient export and for the simulation-based 90% confidence bands in Supplementary Figures S3–S5.


In [3]:
# ==============================================================================
# MULTINOMIAL COMPETING-RISK ESTIMATION
# ==============================================================================

safe_term <- function(x) {
  if (identical(x, "factor(year)")) {
    return(x)
  }
  paste0("`", gsub("`", "", x), "`")
}

prepare_multinomial_model_data <- function(data, controls) {

  formula_terms <- vapply(controls, safe_term, character(1))
  rhs <- if (length(formula_terms) == 0) "1" else paste(formula_terms, collapse = " + ")

  model_frame <- stats::model.frame(
    stats::as.formula(paste("~", rhs)),
    data = data,
    na.action = stats::na.pass,
    drop.unused.levels = FALSE
  )

  keep <- stats::complete.cases(model_frame) &
    !is.na(data$reference) &
    !is.na(data$progress) &
    !is.na(data$failure)

  model_data <- data[keep, , drop = FALSE] %>% droplevels()

  if (nrow(model_data) == 0) {
    return(NULL)
  }

  if (any(model_data$progress == 1 & model_data$failure == 1)) {
    stop("Progression and failure must be mutually exclusive within each project-year.")
  }

  model_data %>%
    dplyr::mutate(
      competing_event = dplyr::case_when(
        progress == 1 ~ "Progress",
        failure == 1 ~ "Failure",
        TRUE ~ "No transition"
      ),
      competing_event = factor(
        competing_event,
        levels = c("No transition", "Progress", "Failure")
      )
    )
}

make_multinomial_formula <- function(controls) {
  formula_terms <- vapply(controls, safe_term, character(1))
  rhs <- if (length(formula_terms) == 0) "1" else paste(formula_terms, collapse = " + ")
  stats::as.formula(paste("competing_event ~", rhs))
}

make_multinomial_coef_names <- function(coef_matrix) {
  unlist(
    lapply(
      rownames(coef_matrix),
      function(event_name) paste0(event_name, "::", colnames(coef_matrix))
    ),
    use.names = FALSE
  )
}

make_clustered_multinomial_vcov <- function(fitted_model, model_data) {
  coef_matrix <- stats::coef(fitted_model)

  if (is.null(dim(coef_matrix)) || nrow(coef_matrix) != 2L) {
    stop("The multinomial model must estimate both Progress and Failure relative to No transition.")
  }

  coefficient_names <- make_multinomial_coef_names(coef_matrix)

  terms_no_response <- stats::delete.response(stats::terms(fitted_model))
  mf <- stats::model.frame(
    terms_no_response,
    data = model_data,
    xlev = fitted_model$xlevels,
    na.action = stats::na.fail,
    drop.unused.levels = FALSE
  )
  x_mat <- stats::model.matrix(
    terms_no_response,
    data = mf,
    contrasts.arg = fitted_model$contrasts
  )

  target_terms <- colnames(coef_matrix)
  missing_terms <- setdiff(target_terms, colnames(x_mat))
  if (length(missing_terms) > 0L) {
    missing_mat <- matrix(0, nrow = nrow(x_mat), ncol = length(missing_terms))
    colnames(missing_mat) <- missing_terms
    x_mat <- cbind(x_mat, missing_mat)
  }
  x_mat <- x_mat[, target_terms, drop = FALSE]

  probabilities <- stats::fitted(fitted_model)
  probabilities <- as.matrix(probabilities)

  required_outcomes <- c("No transition", "Progress", "Failure")
  if (!all(required_outcomes %in% colnames(probabilities))) {
    stop("Fitted multinomial probabilities do not contain all three competing outcomes.")
  }

  y <- model_data$competing_event
  score_blocks <- lapply(
    rownames(coef_matrix),
    function(event_name) {
      residual <- as.numeric(y == event_name) - probabilities[, event_name]
      x_mat * residual
    }
  )

  score_matrix <- do.call(cbind, score_blocks)
  colnames(score_matrix) <- coefficient_names

  cluster_score <- rowsum(
    score_matrix,
    group = as.character(model_data$reference),
    reorder = FALSE
  )

  # nnet::multinom stores the observed Hessian in the same class-by-coefficient
  # ordering used by coef(). Its inverse is the model-based covariance ("bread").
  hessian <- fitted_model$Hessian
  if (is.null(hessian) || any(!is.finite(hessian))) {
    stop("A finite multinomial Hessian is required for clustered inference.")
  }

  bread <- tryCatch(
    solve(hessian),
    error = function(e) MASS::ginv(hessian)
  )

  if (!all(dim(bread) == c(length(coefficient_names), length(coefficient_names)))) {
    stop("The multinomial Hessian dimension does not match the coefficient vector.")
  }

  meat <- crossprod(cluster_score)
  clustered_vcov <- bread %*% meat %*% bread

  n_obs <- nrow(model_data)
  n_clusters <- nrow(cluster_score)
  n_parameters <- length(coefficient_names)

  if (n_clusters > 1L && n_obs > n_parameters) {
    clustered_vcov <- clustered_vcov *
      (n_clusters / (n_clusters - 1)) *
      ((n_obs - 1) / (n_obs - n_parameters))
  }

  dimnames(clustered_vcov) <- list(coefficient_names, coefficient_names)
  clustered_vcov
}

fit_multinomial_competing_model <- function(
    model_data,
    controls,
    transition_name,
    model_name
) {
  model_formula <- make_multinomial_formula(controls)

  event_counts <- table(model_data$competing_event)
  if (any(event_counts[c("No transition", "Progress", "Failure")] == 0L)) {
    stop(
      paste0(
        "All three competing outcomes must be observed for ",
        transition_name,
        " / ",
        model_name,
        ". Counts: ",
        paste(names(event_counts), as.integer(event_counts), sep = "=", collapse = ", ")
      )
    )
  }

  fit_warnings <- character()
  fit_error <- NA_character_

  fitted_model <- tryCatch(
    withCallingHandlers(
      nnet::multinom(
        formula = model_formula,
        data = model_data,
        Hess = TRUE,
        trace = FALSE,
        maxit = 1000,
        MaxNWts = 100000
      ),
      warning = function(w) {
        fit_warnings <<- c(fit_warnings, conditionMessage(w))
        invokeRestart("muffleWarning")
      }
    ),
    error = function(e) {
      fit_error <<- conditionMessage(e)
      NULL
    }
  )

  if (is.null(fitted_model)) {
    stop(
      paste0(
        "The multinomial competing-risk model failed for ",
        transition_name,
        " / ",
        model_name,
        ": ",
        fit_error
      )
    )
  }

  coef_matrix <- stats::coef(fitted_model)
  coefficient_vector <- as.vector(t(coef_matrix))
  names(coefficient_vector) <- make_multinomial_coef_names(coef_matrix)

  clustered_vcov <- make_clustered_multinomial_vcov(
    fitted_model = fitted_model,
    model_data = model_data
  )

  # Store all prediction/inference information with the fitted object so the
  # figure notebooks can calculate coherent probabilities without refitting.
  fitted_model$multinomial_coef_matrix <- coef_matrix
  fitted_model$multinomial_coef_vector <- coefficient_vector
  fitted_model$cluster_vcov <- clustered_vcov
  fitted_model$multinomial_outcome_levels <- c("No transition", "Progress", "Failure")
  fitted_model$multinomial_nonbaseline_levels <- rownames(coef_matrix)
  fitted_model$multinomial_transition <- transition_name
  fitted_model$multinomial_model_name <- model_name

  list(
    model = fitted_model,
    data = model_data,
    transition = transition_name,
    model_name = model_name,
    n_project_years = nrow(model_data),
    n_projects = dplyr::n_distinct(model_data$reference),
    n_no_transition = sum(model_data$competing_event == "No transition"),
    n_progressions = sum(model_data$competing_event == "Progress"),
    n_failures = sum(model_data$competing_event == "Failure"),
    fit_warnings = unique(fit_warnings),
    converged = identical(fitted_model$convergence, 0L)
  )
}

# ------------------------------------------------------------------------------
# Estimate the two specifications on one common stage-specific sample
# ------------------------------------------------------------------------------

multinomial_models <- purrr::imap(
  stage_specs,
  function(stage_value, transition_name) {
    stage_data <- cloglog_data %>%
      dplyr::filter(prev_state == stage_value)

    common_model_data <- prepare_multinomial_model_data(
      data = stage_data,
      controls = common_controls
    )

    if (is.null(common_model_data)) {
      stop(paste0("No complete observations were available for ", transition_name, "."))
    }

    purrr::imap(
      competing_control_steps,
      function(controls, model_name) {
        fit_multinomial_competing_model(
          model_data = common_model_data,
          controls = controls,
          transition_name = transition_name,
          model_name = model_name
        )
      }
    )
  }
)

multinomial_competing_results <- list(
  models = multinomial_models,
  grouped_model_name = grouped_model_name,
  any_enduse_model_name = any_enduse_model_name,
  model_type = "Discrete-time multinomial competing-risk model",
  outcome_levels = c("No transition", "Progress", "Failure"),
  covariance = "Project-clustered sandwich",
  governance_control = governance_control
)


saveRDS(
  multinomial_competing_results,
  "multinomial_competing_risk_results.rds"
)


## Model diagnostics and coefficient tables

The diagnostics report the common stage-specific sample sizes, counts of all three annual outcomes, convergence status and captured fitting warnings. The coefficient table uses the project-clustered covariance matrix stored with each multinomial fit.


In [4]:
# ==============================================================================
# MODEL DIAGNOSTICS AND COEFFICIENT EXPORT
# ==============================================================================

extract_multinomial_coefficient_table <- function(model_obj) {
  coefficient_vector <- model_obj$model$multinomial_coef_vector
  vcov_mat <- model_obj$model$cluster_vcov

  standard_error <- sqrt(diag(vcov_mat))
  z_value <- coefficient_vector / standard_error
  p_value <- 2 * stats::pnorm(abs(z_value), lower.tail = FALSE)

  term_parts <- strsplit(names(coefficient_vector), "::", fixed = TRUE)

  tibble::tibble(
    transition = model_obj$transition,
    model = model_obj$model_name,
    outcome = vapply(term_parts, `[[`, character(1), 1),
    term = vapply(
      term_parts,
      function(x) paste(x[-1], collapse = "::"),
      character(1)
    ),
    estimate = unname(coefficient_vector),
    std_error = unname(standard_error),
    z_value = unname(z_value),
    p_value = unname(p_value)
  )
}

multinomial_coefficient_table <- purrr::map_dfr(
  multinomial_models,
  function(stage_models) {
    purrr::map_dfr(stage_models, extract_multinomial_coefficient_table)
  }
)

multinomial_model_statistics <- purrr::map_dfr(
  multinomial_models,
  function(stage_models) {
    purrr::map_dfr(
      stage_models,
      function(model_obj) {
        tibble::tibble(
          transition = model_obj$transition,
          model = model_obj$model_name,
          n_project_years = model_obj$n_project_years,
          n_projects = model_obj$n_projects,
          n_no_transition = model_obj$n_no_transition,
          n_progressions = model_obj$n_progressions,
          n_failures = model_obj$n_failures,
          converged = model_obj$converged,
          n_fit_warnings = length(model_obj$fit_warnings)
        )
      }
    )
  }
)

print(multinomial_model_statistics)
print(multinomial_coefficient_table, n = Inf)

utils::write.csv(
  multinomial_coefficient_table,
  "multinomial_competing_risk_coefficients.csv",
  row.names = FALSE
)

utils::write.csv(
  multinomial_model_statistics,
  "multinomial_competing_risk_model_statistics.csv",
  row.names = FALSE
)


# A tibble: 6 × 9
  transition     model n_project_years n_projects n_no_transition n_progressions
  <chr>          <chr>           <int>      <int>           <int>          <int>
1 Concept -> Pr… Core…            1724        761            1505            116
2 Concept -> Pr… Core…            1724        761            1505            116
3 Feasibility -… Core…            1966        834            1735             86
4 Feasibility -… Core…            1966        834            1735             86
5 FID -> Progre… Core…             503        268             373             81
6 FID -> Progre… Core…             503        268             373             81
# ℹ 3 more variables: n_failures <int>, converged <lgl>, n_fit_warnings <int>


# A tibble: 228 × 8
    transition         model outcome term  estimate std_error  z_value   p_value
    <chr>              <chr> <chr>   <chr>    <dbl>     <dbl>    <dbl>     <dbl>
  1 Concept -> Progre… Core… Progre… (Int… -1.58e+0    0.490   -3.23   1.22e-  3
  2 Concept -> Progre… Core… Progre… cap.… -3.38e-1    0.108   -3.14   1.69e-  3
  3 Concept -> Progre… Core… Progre… prev… -2.23e-1    0.148   -1.51   1.32e-  1
  4 Concept -> Progre… Core… Progre… tech… -1.16e+0    0.349   -3.32   9.05e-  4
  5 Concept -> Progre… Core… Progre… tech… -3.44e-1    0.485   -0.709  4.78e-  1
  6 Concept -> Progre… Core… Progre… elec… -3.98e-2    0.242   -0.165  8.69e-  1
  7 Concept -> Progre… Core… Progre… endu…  5.31e-1    0.245    2.16   3.05e-  2
  8 Concept -> Progre… Core… Progre… endu…  2.05e-1    0.274    0.747  4.55e-  1
  9 Concept -> Progre… Core… Progre… endu…  6.65e-1    0.270    2.46   1.39e-  2
 10 Concept -> Progre… Core… Progre… endu…  1.56e-1    0.588    0.265  7.91e-  1
 11 Conc